In [ ]:
!pip install -q amplpy
from amplpy import tools
ampl = tools.ampl_notebook(
    modules=["highs", "coin"],
    license_uuid="bcc3d88a-8b8c-4c93-8f21-c2bedd3fc48f")

Licensed to AMPL Community Edition License for <santi.everton@gmail.com>.


# Avaliação Parcial 1 - Unidade 1 (30%)

1. Atividade em equipes de até 3 pessoas;
2. Completar o Notebook, baixar arquivo .ipynb e enviar via SIGAA ao professor até o dia 08/04/2024 23:59:00


## Equipe

* Membro #1: *nome* -  *email*
* Membro #2: *nome* -  *email*
* Membro #3: *nome* -  *email*

## Objetivos

Nesta tarefa os alunos deverão fornecer um programa em Python que usa a API do AMPL para obter a solução de um problema cujo modelo é dado para um conjunto de instâncias da literatura. Os experimentos para estas instâncias deverão ser feitos no servidor da turma e os resultados deverão ser apresentados em formato de tabela como indicado no modelo ao final deste material. As especificações para os experimentos também serão serão fornecidas.


### O Problema do Caixeiro Viajante

Para compreender do que este problema trata, considere o cenário descrito a seguir.

Você está assessorando um candidato ao Governo do Estado do Rio Grande do Norte. A equipe do candidato decidiu que nos próximos dias o candidato deveria participar de uma série de eventos. Estes eventos foram pensados de forma que o candidato visite as 10 cidades mais populosas do estado.

Sabendo o quão cara uma campanha eleitoral pode ser, a equipe decidiu recorrer a especialistas em otimização para que estes pudessem auxiliar na criação de uma rota que especifique a sequência na qual estas cidades deveriam ser visitadas. A comitiva do candidato sairá de Natal, passará por cada cidade uma única vez e, ao final, retornará à Natal, perfazendo a menor quilometragem possível.

Adotar esta medida poderá:

* conomizar combustível;
* economizar tempo;
* reduzir o estresse e fadiga da equipe, bem como do candidato, por realizar os eventos possivelmente em um tempo menor;
* reduzir o desgaste e desvalorização dos veículos utilizados;
* reduzir custos com mão-de-obra, já que seria necessário contratar motoristas para dirigir por um menor tempo;

Reflita quais outros possíveis ganhos a equipe teria ao recorrer à otimização da rota de viagem.

O cenário descreve o Problema do Caixeiro Viajante, ou *Travelling Salesman Problem* - TSP, um dos problemas mais clássicos da matemática.

Este vídeo pode ser útil para melhorar sua compreensão https://www.youtube.com/watch?v=_vKMyRj855A


## Modelando o TSP

Uma das formulações de Programação Linear Inteira mais famosas para o TSP é a formulação de Miller–Tucker–Zemlin (MTZ). Embora seja de fácil entendimento, esta formulação não é tão eficiente quanto outras disponíveis na literatura. A formulação MTZ pode, no entanto, ser aplicada a casos do problema que envolvem poucas entidades, como o cenário que foi apresentado no início deste material.

#### Parâmetros

* $n$ - quantidade de cidades;
* $d_{ij}$ - distância entre as cidades $i$ e $j$, para todo $i,j=1,\ldots,n$;

#### Variáveis de decisão

* $x_{ij}$ - assume valor 1 se ao percorrer a rota devemos nos deslocar da cidade $i$ para a cidade $j$, e zero caso contrário, para todo $i,j=1, \ldots, n$;

* $u_{i}$ - um inteiro contido no conjunto discreto $\{1, 2, \ldots, n-1\}$, para todo $i=2, \ldots, n$. A variável $u_{i}$ é utilizada para sinalizar em qual passo da rota uma cidade $i$ é visitada. Por exemplo, se $u_3=4$, isto significa que a cidade 3 foi a quarta a ser visitada. Note que a cidade de onde partimos, e para onde voltamos ao final da rota, será sempre a cidade 1. Logo, não precisamos de uma variável $u_1$.

#### Formulação MTZ

Utilizando os parâmetros e variáveis de decisão listados, o TSP é formulado como:

$$
\text{minimize}~Z~\sum_{i=1}^{n}\sum_{j=1,j \neq i}^{n}d_{ij}x_{ij} \tag{1}
$$

Sujeito a:

$$
\sum_{i=1,i \neq j}^{n}x_{ij} = 1,~\forall~j=1,2,\ldots,n; \tag{2}
$$

$$
\sum_{j=1,j \neq i}^{n}x_{ij} = 1,~\forall~i=1,2,\ldots,n; \tag{3}
$$

$$
u_i - u_j + (n-1)x_{ij} \leq n-2,~\forall~i,j=2,3,\ldots,n;~ i \neq j \tag{4}
$$

$$
1 \leq u_i \leq n-1,~\forall~i=2,3,\ldots,n; \tag{5}
$$

$$
x_{ij} \in \{0, 1\},~\forall~i,j=1,2,\ldots,n; \tag{6}
$$

$$
u_i \in \mathbb{Z},~\forall~i=2, 3, \ldots, n; \tag{7}
$$

Em que (1) minimiza a distância total da rota gerada. As igualdades em (2) garantem que toda a cidade $j$ deverá ser visitada a partir de uma única cidade $i$, enquanto que (3) garantem ao sair de uma cidade $i$ só poderemos ir a exatamente uma cidade $j$. As desigualdades em (4-5) garantem que quando há um deslocamento de uma cidade $i$ para uma cidade $j$, a cidade $i$ deve ter sido visitada a partir de outra cidade em um momento anterior a este deslocamento para $j$. Por fim, (6-7) são as restrições de domínio sobre os valores das variáveis de decisão.

As restrições dadas em (4) são necessárias para que a solução gerada pelo resolvedor não contenha mais de uma rota, cada uma contemplando apenas parte das cidades. Por exemplo, caso resolvêssemos o modelo excuindo as restrições (4), poderíamos observar uma solução em $x$ como:

$$
x = \begin{pmatrix}
    0 & 0 & 1 & 0 \\
    0 & 0 & 0 & 1 \\
    1 & 0 & 0 & 0 \\
    0 & 1 & 0 & 0
    \end{pmatrix}
$$

Note que solução mostrada em $x$ está dizendo que devemos nos deslocar de 1 para 3 e 3 para 1. A solução também nos diz que devemos nos deslocar de 2 para 4 e de 4 para 2. Logo, temos duas subrotas que passam por duas cidades, e não uma única rota passando por todas as quatro. Ao aplicar as restrições em (4) para os pares de cidades (2,4) e (4,2) e considerando que $u_2=3$ e $u_4=4$, teríamos respectivamente:

$$
-1 + 3 \leq 2
$$

$$
1 + 3 \leq 2
$$

Isto é, a segunda restrição não é satisfeita. Logo, a rota mostrada para $x$ é inviável na formulação MTZ. Ao utilizar as restrições em (4), garantimos que jamais passaremos por uma cidade por mais de uma vez, a não ser que esta cidade seja a origem da rota, portanto a cidade 1. Note que não há restrições em (4) para a cidade 1. Para o exemplo mostrado, note também que não há valores possíveis para $u_i$ que satisfaçam as restrições em (4).


## Tarefa

### Etapa 1

Escreva a formulação MTZ em AMPL, gerando um arquivo .mod

In [ ]:
"""
Seu código aqui
"""

### Etapa 2

Apresente um programa na linguagem de programação de sua preferência para calcular/obter os valores dos parâmetros do TSP a partir das instâncias disponíveis em https://www.math.uwaterloo.ca/tsp/world/countries.html. Considere tratar as instâncias:

1. Djibouti
2. Luxembourg
3. Oman
4. Uruguay
5. Argentina
6. Japan

Note que para estas instâncias o site fornece o valor da solução ótima, que pode ser utilizada para conferência. Use a instância #1 como referência para testes, para só depois resolver as demais. Sugere-se que seu programa receba o nome do arquivo com os dados e retorne alguma estrutura de dados com os valores dos parâmetros $n$ e $d$ (ver formulação).

In [ ]:
"""
Seu código aqui
"""

## Etapa 3

Apresente um programa que por meio de argumentos do terminal recebe o nome de um arquivo contendo uma instância do TSP, um limite de tempo em segundos e um valor limite para o número de threads, resolvendo o TSP meio da API do AMPL via Python. Seu programa deverá permitir execução pela linha de comando em formato similar a:

```shell
python meucodigo.py instancia1.tsp 120 2
```

In [ ]:
"""
Seu código aqui
"""

### Etapa 4

Gere um arquivo Shell Script que automatiza a execução de seu programa para todas as intâncias consideradas na tarefa, gravando em arquivos separados a saída da tela para cada instância. Veja as configurações passadas ao solver no código da aula sobre Localização de Facilidades para que sejam mostrados diversos dados sobre o processo de solução.

In [ ]:
"""
Seu código aqui
"""

### Etapa 5

Realize o seguinte experimento no servidor: resolver o TSP para cada instância, com tempo limite de 3h e 1 thread.

Obtenha os seguintes dados:

### Etapa 6

Reporte os resultados do experimento na tabela mostrada abaixo:

|Instância|Tempo Total|Custo Solução|Limitante Inferior| GAP |Status|
|---|---|---|---|---|---|
|Nome do arquivo|segundos|solução encontrada|melhor estimativa|em %|Ótimo/Viável |

In [13]:
from itertools import product
for solucao in product([0, 1], repeat=20):
    print(solucao)

A saída de streaming foi truncada nas últimas 5000 linhas.
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1)
(1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0)
(1, 1, 1, 1, 1, 1, 1, 0, 1